In [21]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from sklearn.ensemble import RandomForestClassifier

In [22]:
agendamentos = pd.read_csv(
    '../../data/data_interna/agendamentos.csv',
    sep=',',
    encoding='latin1',
    low_memory=False
)

agendamentos.columns = (
    agendamentos.columns
    .str.replace('=\"', '', regex=False)
    .str.replace('\"', '', regex=False)
    .str.strip()
)

print(f'Linhas: {agendamentos.shape[0]}')
print(f'Colunas: {agendamentos.shape[1]}')

agendamentos.head()

Linhas: 333418
Colunas: 93


,cd_agenda_central,hr_agenda,cd_paciente,nm_paciente,vl_altura,qt_peso,dt_nascimento,sn_atendido,sn_encaixe,cd_grupo_agenda,...,cd_conselho,cd_prestador_externo,cd_cid_atend,cd_instituicao,cd_clinica_solicitante,cd_identidade_genero,cd_orientacao_sexual,nm_cidade,cd_uf,dt_hr_agenda
0,174908,23.03.2025 12:09:00,443067,"=""ALFREDO DOS SANTOS""",NaN,NaN,31.12.1950 02:00:00,"=""N""","=""S""",NaN,...,NaN,NaN,"=""""",NaN,NaN,NaN,NaN,"=""NOVA INDEPENDENCIA""","=""SP""","=""23/03/2025 12:09:00"""
1,293733,03.11.2025 10:00:00,1276236,"=""CRISTINA ROBERTA ALVES DA COSTA""",NaN,NaN,08.02.1974 02:00:00,"=""S""","=""N""",NaN,...,NaN,NaN,"=""""",NaN,NaN,NaN,NaN,"=""SALTO DE PIRAPORA""","=""SP""","=""03/11/2025 10:00:00"""
2,293734,26.01.2026 08:00:00,1184837,"=""MARIA AUGUSTA VIEIRA DA SILVA MARTINS""",NaN,NaN,13.03.1965 02:00:00,"=""S""","=""N""",NaN,...,NaN,NaN,"=""""",NaN,NaN,NaN,NaN,"=""VOTORANTIM""","=""SP""","=""26/01/2026 08:00:00"""
3,293558,02.02.2026 08:00:00,1156070,"=""SHEILA MARCIANO ROZA""",NaN,NaN,06.12.1978 03:54:44,"=""N""","=""N""",NaN,...,NaN,NaN,"=""""",NaN,NaN,NaN,NaN,"=""S?O ROQUE""","=""SP""","=""02/02/2026 08:00:00"""
4,293676,23.02.2026 08:00:00,1184837,"=""MARIA AUGUSTA VIEIRA DA SILVA MARTINS""",NaN,NaN,13.03.1965 02:00:00,"=""S""","=""N""",NaN,...,NaN,NaN,"=""""",NaN,NaN,NaN,NaN,"=""VOTORANTIM""","=""SP""","=""23/02/2026 08:00:00"""


In [23]:
agendamentos = agendamentos.dropna(axis=1, how='all')

pct_nulos = agendamentos.isnull().mean()

colunas_validas = pct_nulos[pct_nulos < 0.95].index

agendamentos = agendamentos[colunas_validas]

print(f'Colunas restantes: {agendamentos.shape[1]}')

Colunas restantes: 59


In [24]:
agendamentos['sn_atendido'] = (
    agendamentos['sn_atendido']
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace('=\"', '', regex=False)
    .str.replace('\"', '', regex=False)
)

agendamentos['faltou'] = agendamentos['sn_atendido'].map({
    'N': 1,
    'S': 0
})

agendamentos = agendamentos[
    agendamentos['faltou'].notnull()
]

print(agendamentos['faltou'].value_counts())

faltou
0    232666
1    100752
Name: count, dtype: int64


In [25]:
features = [
    'dt_nascimento',
    'tp_sexo',
    'nm_cidade',
    'cd_uf',
    'tp_forma_agendamento',
    'sn_encaixe',
    'hr_agenda',
    'cd_convenio',
    'cd_con_pla',
    'cd_sub_plano',
    'tp_situacao',
    'sn_publico',
    'dt_gravacao',
    'faltou'
]

features = [
    col for col in features
    if col in agendamentos.columns
]

base = agendamentos[features].copy()

base.head()

,dt_nascimento,tp_sexo,nm_cidade,cd_uf,tp_forma_agendamento,sn_encaixe,hr_agenda,cd_convenio,cd_con_pla,cd_sub_plano,tp_situacao,sn_publico,dt_gravacao,faltou
0,31.12.1950 02:00:00,"=""M""","=""NOVA INDEPENDENCIA""","=""SP""","=""""","=""S""",23.03.2025 12:09:00,2,1.0,"=""""","=""A""","=""S""",28.07.2022 08:08:35,1
1,08.02.1974 02:00:00,"=""F""","=""SALTO DE PIRAPORA""","=""SP""","=""""","=""N""",03.11.2025 10:00:00,165,1.0,"=""""","=""E""","=""S""",31.10.2025 08:37:51,0
2,13.03.1965 02:00:00,"=""F""","=""VOTORANTIM""","=""SP""","=""""","=""N""",26.01.2026 08:00:00,165,1.0,"=""""","=""E""","=""S""",26.01.2026 07:40:13,0
3,06.12.1978 03:54:44,"=""F""","=""S?O ROQUE""","=""SP""","=""P""","=""N""",02.02.2026 08:00:00,165,1.0,"=""""","=""""","=""S""",02.02.2026 07:54:19,1
4,13.03.1965 02:00:00,"=""F""","=""VOTORANTIM""","=""SP""","=""""","=""N""",23.02.2026 08:00:00,165,1.0,"=""""","=""E""","=""S""",19.02.2026 13:39:18,0


In [45]:
base['nm_cidade'] = (
    base['nm_cidade']
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace('=\"', '', regex=False)
    .str.replace('\"', '', regex=False)
)

base['nm_cidade'].head()

0    NOVA INDEPENDENCIA
1     SALTO DE PIRAPORA
2            VOTORANTIM
3             S?O ROQUE
4            VOTORANTIM
Name: nm_cidade, dtype: str

In [6]:
base['dt_nascimento'] = pd.to_datetime(
    base['dt_nascimento'],
    errors='coerce',
    dayfirst=True
)

base['idade'] = (
    (pd.Timestamp.today() - base['dt_nascimento'])
    .dt.days // 365
)

In [26]:
base['tp_sexo'] = (
    base['tp_sexo']
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace('=\"', '', regex=False)
    .str.replace('\"', '', regex=False)
)

base['tp_sexo'] = base['tp_sexo'].replace({
    'MASCULINO': 'M',
    'FEMININO': 'F'
})

base.loc[
    ~base['tp_sexo'].isin(['M', 'F']),
    'tp_sexo'
] = np.nan

print(base['tp_sexo'].value_counts(dropna=False))

tp_sexo
F      181358
M      151842
NaN       218
Name: count, dtype: int64


In [29]:
base['dt_nascimento'] = pd.to_datetime(
    base['dt_nascimento'],
    errors='coerce',
    dayfirst=True
)

base['idade'] = (
    (pd.Timestamp.today() - base['dt_nascimento'])
    .dt.days // 365
)

# remover idades inválidas
base.loc[
    (base['idade'] < 0) |
    (base['idade'] > 110),
    'idade'
] = np.nan

In [30]:
base['hr_agenda'] = (
    base['hr_agenda']
    .astype(str)
    .str.extract(r'(\d{2})')
)

base['hora_consulta'] = pd.to_numeric(
    base['hr_agenda'],
    errors='coerce'
)

In [31]:
base['dt_gravacao'] = pd.to_datetime(
    base['dt_gravacao'],
    errors='coerce',
    dayfirst=True
)

base['dias_antecedencia'] = (
    pd.Timestamp.today() - base['dt_gravacao']
).dt.days

In [32]:
base = base.drop(columns=[
    'dt_nascimento',
    'hr_agenda',
    'dt_gravacao'
], errors='ignore')

base = base.drop_duplicates()

In [33]:
X = base.drop(columns='faltou')

y = base['faltou']

print(X.shape)
print(y.shape)

(291598, 13)
(291598,)


In [34]:
cat_cols = X.select_dtypes(
    include='object'
).columns.tolist()

num_cols = X.select_dtypes(
    exclude='object'
).columns.tolist()

print('CATEGÓRICAS:')
print(cat_cols)

print('\nNUMÉRICAS:')
print(num_cols)

CATEGÓRICAS:
['tp_sexo', 'nm_cidade', 'cd_uf', 'tp_forma_agendamento', 'sn_encaixe', 'cd_sub_plano', 'tp_situacao', 'sn_publico']

NUMÉRICAS:
['cd_convenio', 'cd_con_pla', 'idade', 'hora_consulta', 'dias_antecedencia']


C:\Users\Marco\AppData\Local\Temp\ipykernel_7928\656932844.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(


In [35]:
preprocessor = ColumnTransformer([
    (
        'num',
        Pipeline([
            ('imputer', SimpleImputer(strategy='median'))
        ]),
        num_cols
    ),
    (
        'cat',
        Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]),
        cat_cols
    )
])

In [36]:
modelo = Pipeline([
    ('prep', preprocessor),
    (
        'model',
        RandomForestClassifier(
            n_estimators=200,
            max_depth=12,
            min_samples_leaf=5,
            random_state=42,
            class_weight='balanced'
        )
    )
])

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(233278, 13)
(58320, 13)


In [38]:
modelo.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains s

In [40]:

pred = modelo.predict(X_test)

pred_prob = modelo.predict_proba(X_test)[:, 1]

In [48]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    balanced_accuracy_score,
    matthews_corrcoef
)

print('==============================')
print('MÉTRICAS DO MODELO')
print('==============================\n')

accuracy = accuracy_score(y_test, pred)

precision = precision_score(y_test, pred)

recall = recall_score(y_test, pred)

f1 = f1_score(y_test, pred)

roc_auc = roc_auc_score(y_test, pred_prob)

balanced_acc = balanced_accuracy_score(y_test, pred)

mcc = matthews_corrcoef(y_test, pred)


print(f'Accuracy: {accuracy:.4f}')

print(f'Precision: {precision:.4f}')

print(f'Recall: {recall:.4f}')

print(f'F1-Score: {f1:.4f}')

print(f'ROC AUC: {roc_auc:.4f}')

print(f'Balanced Accuracy: {balanced_acc:.4f}')

print(f'Matthews Corrcoef: {mcc:.4f}')

print('\n==============================')
print('MATRIZ DE CONFUSÃO')
print('==============================\n')

tn, fp, fn, tp = confusion_matrix(
    y_test,
    pred
).ravel()

print(f'True Negatives  (acertou presença): {tn}')

print(f'False Positives (previu falta mas compareceu): {fp}')

print(f'False Negatives (previu presença mas faltou): {fn}')

print(f'True Positives  (acertou falta): {tp}')

print('\n==============================')
print('CLASSIFICATION REPORT')
print('==============================\n')

print(classification_report(y_test, pred))

print('\n==============================')
print('INTERPRETAÇÃO')
print('==============================\n')

MÉTRICAS DO MODELO

Accuracy: 0.8687
Precision: 0.6867
Recall: 0.9996
F1-Score: 0.8141
ROC AUC: 0.9813
Balanced Accuracy: 0.9077
Matthews Corrcoef: 0.7482

MATRIZ DE CONFUSÃO

True Negatives  (acertou presença): 33896
False Positives (previu falta mas compareceu): 7651
False Negatives (previu presença mas faltou): 6
True Positives  (acertou falta): 16767

CLASSIFICATION REPORT

              precision    recall  f1-score   support

           0       1.00      0.82      0.90     41547
           1       0.69      1.00      0.81     16773

    accuracy                           0.87     58320
   macro avg       0.84      0.91      0.86     58320
weighted avg       0.91      0.87      0.87     58320


INTERPRETAÇÃO



In [42]:
rf = modelo.named_steps['model']

feature_names = modelo.named_steps[
    'prep'
].get_feature_names_out()

importancias = pd.DataFrame({
    'feature': feature_names,
    'importance': rf.feature_importances_
})

importancias = importancias.sort_values(
    by='importance',
    ascending=False
)

importancias.head(20)

,feature,importance
2346,"cat__tp_situacao_=""E""",0.341403
2343,"cat__tp_situacao_=""""",0.281067
2344,"cat__tp_situacao_=""A""",0.153880
3,num__hora_consulta,0.044920
0,num__cd_convenio,0.031738
1440,"cat__sn_encaixe_=""N""",0.016737
4,num__dias_antecedencia,0.016032
1434,"cat__cd_uf_=""SP""",0.014549
1441,"cat__sn_encaixe_=""S""",0.013526
1,num__cd_con_pla,0.012896


In [46]:
print('==============================')
print('PERFIL COM MAIOR TENDÊNCIA DE FALTA')
print('==============================\n')


faixas = [0,18,30,45,60,100]

labels = [
    '0-18 anos',
    '19-30 anos',
    '31-45 anos',
    '46-60 anos',
    '60+ anos'
]

base['faixa_etaria'] = pd.cut(
    base['idade'],
    bins=faixas,
    labels=labels
)

perfil_idade = (
    base.groupby('faixa_etaria')['faltou']
    .mean()
    .sort_values(ascending=False)
)

idade_critica = perfil_idade.index[0]

print(f'Faixa etária mais propensa: {idade_critica}')

perfil_cidade = (
    base.groupby('nm_cidade')['faltou']
    .mean()
    .sort_values(ascending=False)
)

cidade_critica = perfil_cidade.index[0]

taxa_cidade = perfil_cidade.iloc[0] * 100

print(
    f'Cidade com maior taxa de falta: '
    f'{cidade_critica} '
    f'({taxa_cidade:.1f}%)'
)

perfil_sexo = (
    base.groupby('tp_sexo')['faltou']
    .mean()
    .sort_values(ascending=False)
)

sexo_critico = perfil_sexo.index[0]

taxa_sexo = perfil_sexo.iloc[0] * 100

print(
    f'Sexo com maior tendência: '
    f'{sexo_critico} '
    f'({taxa_sexo:.1f}%)'
)

perfil_hora = (
    base.groupby(
        pd.cut(
            base['hora_consulta'],
            bins=[0,6,9,12,15,18,24]
        )
    )['faltou']
    .mean()
    .sort_values(ascending=False)
)

hora_critica = perfil_hora.index[0]

print(
    f'Horário com maior ausência: '
    f'{hora_critica}'
)

PERFIL COM MAIOR TENDÊNCIA DE FALTA

Faixa etária mais propensa: 19-30 anos
Cidade com maior taxa de falta: ADRIANOPOLIS (100.0%)
Sexo com maior tendência: F (28.8%)
Horário com maior ausência: (0, 6]


In [49]:
print('==============================')
print('RESUMO EXECUTIVO')
print('==============================\n')

print(
    f'Pacientes da cidade {cidade_critica}, '
    f'principalmente da faixa etária {idade_critica}, '
    f'apresentam maior tendência de faltar às consultas. '
    f'O perfil identificado também indica maior risco '
    f'para pacientes do sexo {sexo_critico}.'
)

RESUMO EXECUTIVO

Pacientes da cidade ADRIANOPOLIS, principalmente da faixa etária 19-30 anos, apresentam maior tendência de faltar às consultas. O perfil identificado também indica maior risco para pacientes do sexo F.
